In [2]:
import pickle
from pathlib import Path

import numpy as np
import torch
from getdist import MCSamples, plots
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

run_dir = Path("/home/cbllover/HalfDome/SBI_analysis/convergence_tests/N523788")

In [14]:
x_obs_full = np.load(run_dir / "so_masked_baseline_noise_cross_deproj0_ell80_7979_last10.npz")

In [53]:
N_index = 5

In [54]:
import pickle
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from getdist import MCSamples, plots


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

run_dir = Path(run_dir)

dataset_file = (
    run_dir
    / "so_masked_baseline_noise_cross_deproj0_ell80_7979_sbi_run.npz"
)


# ------------------------------------------------------------
# Load trained SBI objects
# ------------------------------------------------------------

with open(run_dir / "inference.pkl", "rb") as f:
    inference = pickle.load(f)

with open(run_dir / "density_estimator.pkl", "rb") as f:
    density_estimator = pickle.load(f)

posterior = inference.build_posterior(density_estimator)


# ------------------------------------------------------------
# Load x transform
# ------------------------------------------------------------

with np.load(run_dir / "x_transform.npz", allow_pickle=True) as data:

    mode = (
        data["mode"].item()
        if "mode" in data.files
        else "none"
    )

    mean = (
        np.asarray(data["mean"], dtype=np.float32)
        if "mean" in data.files
        else None
    )

    std = (
        np.asarray(data["std"], dtype=np.float32)
        if "std" in data.files
        else None
    )

    scale = (
        np.asarray(data["scale"], dtype=np.float32)
        if "scale" in data.files
        else None
    )


print("x transform mode:", mode)


def apply_x_transform(x):
    x = np.asarray(x, dtype=np.float32)

    if mode in ("", "none", "raw"):
        return x

    elif mode in ("asinh", "asinh_median_abs"):
        if scale is None:
            raise ValueError("Transform requires 'scale', but none was saved.")
        return np.arcsinh(x / scale)

    elif mode == "standardize":
        if mean is None or std is None:
            raise ValueError(
                "standardize requires saved mean and std."
            )
        return (x - mean) / std

    elif mode == "asinh_standardize":
        if scale is None or mean is None or std is None:
            raise ValueError(
                "asinh_standardize requires scale, mean and std."
            )

        x_asinh = np.arcsinh(x / scale)
        return (x_asinh - mean) / std

    else:
        raise ValueError(
            f"Unsupported x transform mode: {mode!r}"
        )


# ------------------------------------------------------------
# Load the actual observation used by this SBI run
# ------------------------------------------------------------

x_obs = x_obs_full["x"][N_index]
obs_index = x_obs_full["obs_index"]
print("x_obs shape:", x_obs.shape)
print("obs_index:", obs_index)

theta_saved = x_obs_full["theta"][N_index]

# ------------------------------------------------------------
# Transform conditioning vector
# ------------------------------------------------------------

x_cond = apply_x_transform(x_obs)

if x_cond.shape != (40,):
    raise ValueError(
        f"Expected conditioned x shape (40,), got {x_cond.shape}"
    )

if not np.all(np.isfinite(x_cond)):
    raise ValueError(
        "x_cond contains NaN or Inf values."
    )

x_t = torch.as_tensor(
    x_cond,
    dtype=torch.float32,
)


# ------------------------------------------------------------
# Sample posterior
# ------------------------------------------------------------

with torch.no_grad():
    samples = posterior.sample(
        (100_000,),
        x=x_t,
        show_progress_bars=False,
    )

samples = samples.detach().cpu().numpy()
samples = np.asarray(samples, dtype=np.float64)

print("Posterior samples shape:", samples.shape)

if samples.shape[1] != 9:
    raise ValueError(
        f"Expected 9 parameters, got {samples.shape[1]}"
    )


# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------

param_names = [
    "P0",
    "xc",
    "beta",
    "alpha_m_P0",
    "alpha_m_xc",
    "alpha_m_beta",
    "alpha_z_P0",
    "alpha_z_xc",
    "alpha_z_beta",
]

theta_true = theta_saved

labels = [
    r"P_0",
    r"x_c",
    r"\beta",
    r"\alpha_{m,P_0}",
    r"\alpha_{m,x_c}",
    r"\alpha_{m,\beta}",
    r"\alpha_{z,P_0}",
    r"\alpha_{z,x_c}",
    r"\alpha_{z,\beta}",
]


# ------------------------------------------------------------
# Optional sanity check
# ------------------------------------------------------------

if theta_saved is not None:
    print("\nDifference between hard-coded theta_true and saved obs_theta:")
    print(theta_true - theta_saved)


# ------------------------------------------------------------
# GetDist
# ------------------------------------------------------------

gd = MCSamples(
    samples=samples,
    names=param_names,
    labels=labels,
)

gd.updateSettings({
    "smooth_scale_1D": 0.3,
    "smooth_scale_2D": 0.3,
    "fine_bins": 2048,
    "fine_bins_2D": 1024,
})


# ------------------------------------------------------------
# Triangle plot
# ------------------------------------------------------------

g = plots.get_subplot_plotter(
    width_inch=18.0 / 2.54
)

g.settings.axes_fontsize = 7
g.settings.lab_fontsize = 8
g.settings.legend_fontsize = 7
g.settings.alpha_filled_add = 0.32
g.settings.linewidth = 1.0
g.settings.num_plot_contours = 2
g.settings.figure_legend_frame = False
g.settings.scaling = False

g.triangle_plot(
    [gd],
    params=param_names,
    filled=True,
    legend_labels=["SBI (N=523,788)"],
    contour_colors=["#1f77b4"],
    line_args=[
        {
            "color": "#1f77b4",
            "lw": 1.2,
        }
    ],
    markers=dict(zip(param_names, theta_true)),
    marker_args={
        "color": "black",
        "lw": 0.8,
        "ls": ":",
    },
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

outfile = f"convergence_tests/N523788/battaglia12_sbi_corner_N523788_{N_index}.png"

plt.savefig(
    outfile,
    dpi=300,
    bbox_inches="tight",
)

plt.close("all")

print(f"Saved {outfile}")

x transform mode: asinh
x_obs shape: (40,)
obs_index: 524287
Posterior samples shape: (100000, 9)

Difference between hard-coded theta_true and saved obs_theta:
[0. 0. 0. 0. 0. 0. 0. 0. 0.]
Removed no burn in
Saved convergence_tests/N523788/battaglia12_sbi_corner_N523788_5.png


In [55]:
# ============================================================
# Correct parameter ordering + actual priors
# ============================================================

param_names = [
    "P0",
    "xc",
    "beta",
    "alpha_m_P0",
    "alpha_m_xc",
    "alpha_m_beta",
    "alpha_z_P0",
    "alpha_z_xc",
    "alpha_z_beta",
]

prior_low = np.array([
    1.832524,    # P0
    0.150011,    # xc
    3.480627,    # beta
    0.000312,    # alpha_m_P0
   -0.099718,    # alpha_m_xc
   -0.019935,    # alpha_m_beta
   -1.363457,    # alpha_z_P0
    0.147393,    # alpha_z_xc
    0.083808,    # alpha_z_beta
])

prior_high = np.array([
    34.341221,   # P0
    0.844503,    # xc
    5.216611,    # beta
    0.292251,    # alpha_m_P0
    0.099795,    # alpha_m_xc
    0.099767,    # alpha_m_beta
   -0.228839,    # alpha_z_P0
    1.314474,    # alpha_z_xc
    0.745884,    # alpha_z_beta
])

theta_true = theta_saved

# ============================================================
# Posterior statistics
# ============================================================

posterior_mean = np.mean(samples, axis=0)
posterior_std = np.std(samples, axis=0)
posterior_var = np.var(samples, axis=0)

prior_width = prior_high - prior_low


# Normalize to each parameter's actual prior
true_norm = (theta_true - prior_low) / prior_width
mean_norm = (posterior_mean - prior_low) / prior_width
std_norm = posterior_std / prior_width


# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=(13, 7))

y = np.arange(9)

# Full prior ranges
for i in range(9):
    ax.hlines(
        y=i,
        xmin=0,
        xmax=1,
        linewidth=6,
        alpha=0.18,
        zorder=1,
    )

# Posterior mean ± std
ax.errorbar(
    mean_norm,
    y,
    xerr=std_norm,
    fmt="o",
    markersize=6,
    capsize=4,
    linewidth=1.5,
    label=r"Posterior mean $\pm 1\sigma$",
    zorder=3,
)

# True values
ax.scatter(
    true_norm,
    y,
    marker="x",
    s=70,
    linewidths=2,
    label="True",
    zorder=4,
)


# ============================================================
# Numerical information
# ============================================================

for i in range(9):

    text = (
        rf"true={theta_true[i]:.4g}   "
        rf"mean={posterior_mean[i]:.4g}   "
        rf"$\sigma$={posterior_std[i]:.3g}   "
        rf"var={posterior_var[i]:.3g}"
    )

    ax.text(
        1.03,
        i,
        text,
        va="center",
        fontsize=8,
        transform=ax.get_yaxis_transform(),
    )


# ============================================================
# Formatting
# ============================================================

labels = [
    r"$P_0$",
    r"$x_c$",
    r"$\beta$",
    r"$\alpha_{m,P_0}$",
    r"$\alpha_{m,x_c}$",
    r"$\alpha_{m,\beta}$",
    r"$\alpha_{z,P_0}$",
    r"$\alpha_{z,x_c}$",
    r"$\alpha_{z,\beta}$",
]

ax.set_yticks(y)
ax.set_yticklabels(labels)

ax.set_xlim(-0.05, 1.05)

ax.set_xticks([0, 0.25, 0.5, 0.75, 1])

ax.set_xlabel(
    r"Position within prior: "
    r"$(\theta-\theta_{\rm low})/"
    r"(\theta_{\rm high}-\theta_{\rm low})$"
)

ax.set_title("Battaglia12 SBI parameter recovery")

# Explicit prior edges
ax.axvline(
    0,
    linestyle=":",
    linewidth=1,
    label="Prior bounds",
)

ax.axvline(
    1,
    linestyle=":",
    linewidth=1,
)

ax.grid(axis="x", alpha=0.25)

ax.invert_yaxis()

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.10),
    ncol=3,
)

# Room for numerical information
plt.subplots_adjust(
    left=0.13,
    right=0.61,
    bottom=0.17,
    top=0.92,
)

outfile = f"convergence_tests/N523788/battaglia12_true_vs_mean_actual_prior_{N_index}.png"

plt.savefig(
    outfile,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", outfile)

Saved: convergence_tests/N523788/battaglia12_true_vs_mean_actual_prior_5.png
